In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import os

## Feature Engineering & Dataset Construction
In this notebook:
1. Building the exact (X, y) tensors the LSTM will consume
2. Standardize features, construct sliding window sequences
3. Handle class imbalance, wrap in PyTorch Dataset and DataLoader

### Load master dataframe

In [2]:
df = pd.read_csv("data/processed/master.csv", index_col="date", parse_dates=True)
print(f"Loaded master dataframe: {df.shape}")

Loaded master dataframe: (5012, 9)


### TRAIN/TEST/VALIDATE Split

In [3]:
FEATURE_COLS = [
    "sp500_return", "vix_diff", "t10y_diff", "t2y_diff",
    "fedfunds_diff", "credit_diff", "realized_vol", "yield_spread_diff"
]
TARGET_COL   = "regime"
WINDOW       = 20    # lookback window in trading days
TRAIN_RATIO  = 0.7
VAL_RATIO    = 0.15 #validate ratio


n         = len(df)
n_train   = int(n * TRAIN_RATIO)
n_val     = int(n * VAL_RATIO)

train_df  = df.iloc[:n_train]
val_df    = df.iloc[n_train:n_train + n_val]
test_df   = df.iloc[n_train + n_val:]

print(f"\nSplit sizes:")
print(f"Train : {len(train_df):,} rows  ({train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Val: {len(val_df):,} rows  ({val_df.index[0].date()} to {val_df.index[-1].date()})")
print(f"Test: {len(test_df):,} rows  ({test_df.index[0].date()} to {test_df.index[-1].date()})")


Split sizes:
Train : 3,508 rows  (2005-02-02 to 2019-01-09)
Val: 751 rows  (2019-01-10 to 2021-12-31)
Test: 753 rows  (2022-01-03 to 2024-12-31)


### Standardize features
Fit scaler only on training data and never on validation or test set. This helps prevents any future information leaking into feature scaling.

In [4]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[FEATURE_COLS])
val_scaled   = scaler.transform(val_df[FEATURE_COLS])
test_scaled  = scaler.transform(test_df[FEATURE_COLS])

train_labels = train_df[TARGET_COL].values
val_labels   = val_df[TARGET_COL].values
test_labels  = test_df[TARGET_COL].values

# Save scaler for later use
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump(scaler, "models/scaler.pkl")
print(f"\nScaler saved to models/scaler.pkl")


Scaler saved to models/scaler.pkl


### Sliding window sequence construction

In [5]:
def make_sequences(features, labels, window):
    """
    Convert a 2D feature array into 3D sequences for LSTM input.
    Each sample contains 'window' consecutive days of features
    and the label for the day immediately following the window.

    Args:
        features: np.array of shape (n_days, n_features)
        labels: np.array of shape (n_days,)
        window: int, number of past days to include in each sample

    Returns:
        X: np.array of shape (n_samples, window, n_features)
        y: np.array of shape (n_samples,)
    """
    X, y = [], []
    for t in range(window, len(features)):
        X.append(features[t - window:t])   # past 'window' days
        y.append(labels[t])                # label for day t
    return np.array(X), np.array(y)

In [6]:
X_train, y_train = make_sequences(train_scaled, train_labels, WINDOW)
X_val,   y_val = make_sequences(val_scaled,   val_labels,   WINDOW)
X_test,  y_test = make_sequences(test_scaled,  test_labels,  WINDOW)

print(f"\nSequence shapes:")
print(f"X_train: {X_train.shape} y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape} y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape} y_test : {y_test.shape}")


Sequence shapes:
X_train: (3488, 20, 8) y_train: (3488,)
X_val  : (731, 20, 8) y_val  : (731,)
X_test : (733, 20, 8) y_test : (733,)


### Class imbalance: compute class weights

In [7]:
n_calm      = (y_train == 0).sum()
n_turbulent = (y_train == 1).sum()
weight_calm      = 1.0
weight_turbulent = n_calm/n_turbulent

class_weights = torch.tensor([weight_calm, weight_turbulent], dtype=torch.float32)
print(f"\nClass weights for loss function:")
print(f"Calm (0): {weight_calm:.4f}")
print(f"Turbulent (1): {weight_turbulent:.4f}")


Class weights for loss function:
Calm (0): 1.0000
Turbulent (1): 2.3030


### PyTorch Dataset

In [8]:
class RegimeDataset(Dataset):
    """
    PyTorch Dataset wrapping (X, y) sequence arrays.
    X shape: (n_samples, window, n_features)
    y shape: (n_samples,)
    """
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

### PyTorch DataLoaders

In [9]:
BATCH_SIZE = 32

train_dataset = RegimeDataset(X_train, y_train)
val_dataset = RegimeDataset(X_val,   y_val)
test_dataset = RegimeDataset(X_test,  y_test)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) #shuffling only in train set
val_loader = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders ready:")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


DataLoaders ready:
Train batches: 109
Val batches: 23
Test batches: 23


### Sanity check: inspect one batch

In [10]:
X_batch, y_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"X shape: {X_batch.shape} (batch_size, window, n_features)")
print(f"y shape: {y_batch.shape} (batch_size,)")
print(f"y values: {y_batch[:10].tolist()}")


Sample batch:
X shape: torch.Size([32, 20, 8]) (batch_size, window, n_features)
y shape: torch.Size([32]) (batch_size,)
y values: [0, 0, 1, 0, 0, 0, 0, 0, 0, 1]


### Save arrays for use in model fitting

In [11]:
np.save("data/processed/X_train.npy", X_train)
np.save("data/processed/X_val.npy",   X_val)
np.save("data/processed/X_test.npy",  X_test)
np.save("data/processed/y_train.npy", y_train)
np.save("data/processed/y_val.npy",   y_val)
np.save("data/processed/y_test.npy",  y_test)
print(f"\nArrays saved to data/processed/")


Arrays saved to data/processed/
